# Ablation Metrics Plotting

This notebook provides functions for plotting ablation experiment results from the metrics_ablations.json file.

The data structure is:
```
f"za_heads_layers_{list-of-layers-ablated (ints separated by '-')}": {
    prediction_horizon: {
        system_name: {
            metric_name: float
        }
    }
}
```


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from tsfm_lens.utils import apply_custom_style
import os
from tsfm_lens.utils import (
    plot_ablation_metrics_distributional,
    combine_metrics_dicts,
    load_json_data,
)

In [ ]:
apply_custom_style("../../config/plotting.yaml")

In [ ]:
ablations_results_subdirname_dict = {
    "Chronos": "ablations_chronos_amazon-chronos-t5-base",
    "Chronos Bolt": "ablations_chronos_bolt_amazon-chronos-bolt-base",
    "TimesFM 2.5": "ablations_timesfm_google-timesfm-2.5-200m-pytorch",
    "Toto": "ablations_toto_Datadog-Toto-Open-Base-1.0",
    "Moirai": "ablations_moirai_Salesforce-moirai-1.1-R-base",
}

In [ ]:
model_name = "TimesFM 2.5"
ablations_results_subdirname = ablations_results_subdirname_dict[model_name]

In [ ]:
plot_save_dir = os.path.join("../../figures", ablations_results_subdirname)
os.makedirs(plot_save_dir, exist_ok=True)
print(f"Plot save dir: {plot_save_dir}")

## Load the data and visualize

In [ ]:
# Get root path
root_path = os.path.abspath(os.path.join(os.path.dirname("__file__"), "..", ".."))
print(f"Root path: {root_path}")

In [ ]:
rseed = 123
metrics_data_path = os.path.join(root_path, "outputs")
# metrics_data_path = os.path.join(root_path, "outputs_base40")

ablations_results_dir = os.path.join(
    metrics_data_path, ablations_results_subdirname, f"rseed-{rseed}", "ablations_vs_original"
)

# get the specific subdirectory with the num_heads_to_ablate_per_layer setting
n_heads_to_ablate_per_layer = None
if n_heads_to_ablate_per_layer is None:
    n_heads_to_ablate_per_layer_str = "all_heads"
else:
    n_heads_to_ablate_per_layer_str = f"nheads-{n_heads_to_ablate_per_layer}"
ablations_results_subdir = os.path.join(ablations_results_dir, n_heads_to_ablate_per_layer_str)
print(f"Ablations results path: {ablations_results_subdir}")


In [ ]:
dataset_name = "gift-eval"
num_eval_instances = 10  # number of splits/windows per dataset
selected_layergroup = 4  # number of consecutive layers ablated
terms = ["long", "short"]  # for gift-eval
window_start_times = [1512]  # for general datasets

# dataset_name = "base40"
# num_eval_instances = 1  # number of splits/windows per dataset
# selected_layergroup = 2  # number of consecutive layers ablated
# terms = ["long", "short"]  # for gift-eval
# window_start_times = [1512]  # for general datasets

# dataset_name = "base40"
# num_eval_instances = 1  # number of splits/windows per dataset
# selected_layergroup = 2  # number of consecutive layers ablated
# terms = ["long", "short"]  # for gift-eval
# window_start_times = [2512]  # for general datasets

metrics_run_name = f"ablations_vs_original_nwindows{num_eval_instances}_layergroup{selected_layergroup}_rseed{rseed}"

if dataset_name == "gift-eval":
    base_sig = f"metrics_%s_{dataset_name}_nwindows-{num_eval_instances}_za-%s_layergroup-{selected_layergroup}"
    fname_signatures = {
        "Heads": [base_sig % (term, "head") for term in terms],
        "MLPs": [base_sig % (term, "mlp") for term in terms],
        "Heads and MLPs": [base_sig % (term, "head-mlp") for term in terms],
    }
else:
    base_sig = f"metrics_{dataset_name}_start-%s_nwindows-{num_eval_instances}_za-%s_layergroup-{selected_layergroup}"
    fname_signatures = {
        "Heads": [base_sig % (t, "head") for t in window_start_times],
        "MLPs": [base_sig % (t, "mlp") for t in window_start_times],
        "Heads and MLPs": [base_sig % (t, "head-mlp") for t in window_start_times],
    }

print(f"Fname signatures: {fname_signatures}")
all_files = os.listdir(ablations_results_subdir)
print(f"There are {len(all_files)} files in {ablations_results_subdir}")
# filepaths that match any of the fname_signatures
data_paths_dict = {
    k: [os.path.join(ablations_results_subdir, f) for f in all_files if any(sig in f for sig in v)]
    for k, v in fname_signatures.items()
}
print(f"Data paths dict: {data_paths_dict}")
data_dicts_by_component = {}
for component, data_paths in data_paths_dict.items():
    data_fname = data_paths[0].split("/")[-1]
    print(data_fname)
    print(f"Found {len(data_paths)} metrics files for {component}: {data_fname}")
    print("Loading data...")
    data_dicts_by_component[component] = [load_json_data(data_path) for data_path in data_paths]

combined_data_dicts_by_component = {
    component: combine_metrics_dicts(data_dicts, verbose=True)
    for component, data_dicts in data_dicts_by_component.items()
}

### Plot for Single Component Ablations

Available metrics: ['smape', 'wape', 'mase', 'wql', 'spearman', 'pearson', 'ssim', 'energy_distance', 'spectral_hellinger', 'spectral_wasserstein-1', 'spectral_wasserstein-2', 'cross_spectral_phase_similarity']

In [ ]:
save_fig = True
save_path = None

# selected_metric = "cross_spectral_phase_similarity"
# selected_metric = "ssim"
selected_metric = "spearman"

selected_components = "Heads and MLPs"
# selected_components = "Heads"
# selected_components = "MLPs"

In [ ]:
# # For presentation of the Heads and MLPs ablations
# excluded_layer_indices = {
#     "Toto": [1, 3, 4, 6],
#     "TimesFM 2.5": [2, 3, 4, 5, 8, 14, 15, 16],
#     "Chronos Bolt": [3],
#     "Chronos": [3, 6],
#     "Moirai": [5, 1, 2, 10],
# }

# # For presentation of the Heads ablations
# excluded_layer_indices = {
#     "Toto": [2, 3, 4, 6],
#     "TimesFM 2.5": [2, 3, 4, 5, 8, 14, 15, 16],  # [2, 3, 4, 5, 7, 8, 12, 13, 14, 16],
#     "Chronos Bolt": [3],
#     "Chronos": [3, 4, 6],
#     "Moirai": [],
# }

# For presentation of the MLP ablations
excluded_layer_indices = {
    "Toto": [],
    "TimesFM 2.5": [],  # [2, 3, 4, 5, 7, 8, 12, 13, 14, 16],
    "Chronos Bolt": [],
    "Chronos": [],
    "Moirai": [],
}

# hide_indices=[2, 3, 4, 5, 8, 14, 15, 16],  # [2, 3, 4, 5, 7, 8, 12, 13, 14, 16],
# hide_indices=[3],
# hide_indices=[3, 6], # [3, 4, 6],

In [ ]:
for selected_horizon in ["64"]:
    if save_fig:
        save_path = os.path.join(
            plot_save_dir,
            f"layergroup-{selected_layergroup}",
            f"{model_name}_{selected_metric}_{selected_horizon}_{metrics_run_name}_components-{selected_components}.pdf",
        )
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
    print(f"Saving to {save_path}")
    fig = plot_ablation_metrics_distributional(
        combined_data_dicts_by_component[selected_components],
        metric_name=selected_metric,
        prediction_horizon=selected_horizon,
        cmap="Blues",
        # title=rf"ZA ({selected_components}) $L_{{\mathrm{{pred}}}} = {int(selected_horizon)}$",
        # title=f"{model_name} (ZA {selected_components})",
        title=f"{model_name}",
        figsize=(4, 5),
        plot_type="violin",
        show_mean_line=False,
        save_path=save_path,
        percentile_range=(25, 75),
        whisker_range=(0, 95),
        tick_rotation=0 if selected_layergroup == 1 else 45,
        verbose=True,
        hide_indices=excluded_layer_indices[model_name],
        exclude_outliers_iqr=2,
        show_median_trend=True,
    )  #


In [ ]:
# from tsfm_lens.utils import get_layergroup_metrics
# import pandas as pd
# import numpy as np


# def display_ablation_metrics_distributional(
#     data: dict[str, dict[str, dict[str, dict[str, float | list[float]]]]],
#     metric_name: str,
#     prediction_horizon: str,
#     exclude_outliers_iqr: float | None = None,
#     title: str | None = None,
# ) -> "pd.DataFrame":
#     """Print a table of ablation metrics across different layer ablations.

#     Args:
#         data: Ablation metrics data dictionary.
#         metric_name: Name of the metric to display.
#         prediction_horizon: Prediction horizon to filter by.
#         hide_indices: List of x-axis indices (0-based) to exclude from the table.
#         exclude_outliers_iqr: If set, exclude outliers using IQR method.
#             Values outside [Q1 - multiplier*IQR, Q3 + multiplier*IQR] are removed.
#             Common values: 1.5 (standard), 3.0 (extreme outliers only).
#         title: Optional title for the table.

#     Returns:
#         DataFrame with the metrics table.
#     """
#     layer_group_metrics, x_labels = get_layergroup_metrics(data, metric_name, prediction_horizon)
#     # Exclude IQR outliers
#     if exclude_outliers_iqr is not None:
#         layer_group_metrics = [
#             arr[
#                 (arr >= (q1 := np.percentile(arr, 25)) - exclude_outliers_iqr * (iqr := np.percentile(arr, 75) - q1))
#                 & (arr <= q1 + iqr + exclude_outliers_iqr * iqr)
#             ]
#             for arr in layer_group_metrics
#         ]

#     # Build table data
#     table_data = []
#     for label, arr in zip(x_labels, layer_group_metrics):
#         table_data.append(
#             {
#                 "Ablated Layers": label,
#                 # "Count": len(arr),
#                 "Median": f"{np.median(arr):.2f}",
#                 "Q25": f"{np.percentile(arr, 25):.2f}",
#                 "Q75": f"{np.percentile(arr, 75):.2f}",
#             }
#         )

#     df = pd.DataFrame(table_data)

#     # Display the dataframe
#     # display(df)
#     print(df)

#     return df

In [ ]:
# print(f"Model name: {model_name}")
# for sel_comp in ["Heads and MLPs", "Heads", "MLPs"]:
#     print(f"Under ablations of {sel_comp}...")
#     for sel_met in ["spearman", "cross_spectral_phase_similarity"]:
#         print(f"Metric name: {sel_met}")
#         for selected_horizon in ["64", "512"]:
#             print("=" * 20)
#             print(f"Selected horizon: {selected_horizon}")
#             display_ablation_metrics_distributional(
#                 combined_data_dicts_by_component[sel_comp],
#                 metric_name=sel_met,
#                 prediction_horizon=selected_horizon,
#                 # exclude_outliers_iqr=2,
#             )  #
